# 단일 GPU에서 OOM이 발생하는 14B 이상 모델을 ZeRO-3+Offload로 훈련 성공시키기

> 본 노트북은 SYLLABUS.md에 기반하여 자동 생성된 뼈대(Skeleton) 파일입니다. 상세한 이론, 수식 및 코드는 추가로 구현되어야 합니다.

## 1. 개요 및 학습 목표
이 노트북에서는 해당 주제에 대한 핵심 개념을 다룹니다.

## 2. 핵심 이론 및 수학적 원리
- 수식 및 상세한 동작 원리를 여기에 기록합니다.

## 3. 실습 코드 구현
아래 셀을 통해 파이썬 및 관련 프레임워크 코드를 직접 작성하고 실행해 볼 수 있습니다.

In [ ]:
# 실습을 위한 기본 라이브러리 임포트
import tensorflow as tf
import numpy as np

print(f"TensorFlow Version: {tf.__version__}")


---
## Q1: ZeRO 단계별 메모리 절감 계산

DeepSpeed ZeRO 최적화의 각 단계(Stage 1, 2, 3)에서 GPU당 메모리 사용량을 계산하세요.

**요구사항:**
- 모델 파라미터 수 `N = 7e9`, GPU 수 `num_gpus = 8`로 설정하세요.
- 각 요소별 메모리(bytes):
  - 파라미터: `N * 2` (fp16)
  - 그래디언트: `N * 2` (fp16)
  - Adam 옵티마이저 상태: `N * 12` (fp32: param copy + m + v)
- **Stage 1**: 옵티마이저 상태만 `num_gpus`로 분할
- **Stage 2**: Stage 1 + 그래디언트도 분할
- **Stage 3**: Stage 2 + 파라미터도 분할
- 각 스테이지의 GPU당 메모리(GB)를 출력하세요.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
N = 7e9
num_gpus = 8
bytes_to_gb = 1 / (1024**3)

param_mem = N * 2      # fp16 파라미터
grad_mem = N * 2       # fp16 그래디언트
optim_mem = N * 12     # fp32 Adam (param_copy=4, m=4, v=4)

# Baseline (ZeRO 없음)
baseline = (param_mem + grad_mem + optim_mem) * bytes_to_gb

# ZeRO Stage 1: 옵티마이저 상태만 분할
stage1 = (param_mem + grad_mem + optim_mem / num_gpus) * bytes_to_gb

# ZeRO Stage 2: 옵티마이저 + 그래디언트 분할
stage2 = (param_mem + (grad_mem + optim_mem) / num_gpus) * bytes_to_gb

# ZeRO Stage 3: 모두 분할
stage3 = (param_mem + grad_mem + optim_mem) / num_gpus * bytes_to_gb

print(f"{'단계':<15} {'GPU당 메모리':>12} {'절감률':>10}")
print('-' * 40)
print(f"{'기준 (없음)':<15} {baseline:>11.1f}GB {0:>9.0f}%")
print(f"{'ZeRO Stage 1':<15} {stage1:>11.1f}GB {(1-stage1/baseline)*100:>9.1f}%")
print(f"{'ZeRO Stage 2':<15} {stage2:>11.1f}GB {(1-stage2/baseline)*100:>9.1f}%")
print(f"{'ZeRO Stage 3':<15} {stage3:>11.1f}GB {(1-stage3/baseline)*100:>9.1f}%")
```
</details>

---
## Q2: ZeRO-3 파라미터 파티셔닝 시뮬레이션

ZeRO Stage 3에서 각 GPU가 모델 파라미터의 일부만을 저장하는 파티셔닝을 간단히 시뮬레이션하세요.

**요구사항:**
- `num_workers = 4`, 전체 파라미터를 100개로 가정하세요.
- `partition_params(params, num_workers)` 함수를 정의하여, 파라미터 배열을 `num_workers`로 균등하게 나눠 각 워커에 할당하세요.
- 워커 2가 forward pass에서 파라미터를 필요로 할 때 All-Gather로 전체 파라미터를 복원하는 시뮬레이션을 작성하세요.
- forward 전/후의 메모리 상태(각 워커가 가진 파라미터 수)를 출력하세요.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
import numpy as np

num_workers = 4
total_params = 100
params = np.arange(total_params, dtype=np.float32)

def partition_params(params, num_workers):
    chunk = len(params) // num_workers
    return {i: params[i*chunk:(i+1)*chunk].copy() for i in range(num_workers)}

# 각 워커가 자신 파티션만 보유
partitions = partition_params(params, num_workers)
print("ZeRO-3 파티션 상태:")
for w, p in partitions.items():
    print(f"  워커 {w}: 파라미터 {len(p)}개 (idx {p[0]:.0f}~{p[-1]:.0f})")

# Forward pass 시 All-Gather (워커 2가 전체 파라미터 필요)
local_worker = 2
all_gather = np.concatenate([partitions[i] for i in range(num_workers)])
print(f"\n워커 {local_worker} All-Gather 후 전체 파라미터: {len(all_gather)}개")
print(f"복원 정확성: {np.allclose(all_gather, params)}")

# Forward 후 해제 (ZeRO-3: 쓰고 나면 메모리 해제)
print(f"\nForward 완료 → 워커 {local_worker}가 비소유 파라미터 해제 (시뮬레이션)")
print(f"해제 후 워커 {local_worker} 보유 파라미터: {len(partitions[local_worker])}개")
```
</details>

---
## Q3: Gradient Checkpointing 메모리 절감 비교

Gradient Checkpointing은 활성화값을 backward 시 재계산하여 메모리를 절약하는 기법입니다. 활성화 메모리 절감량을 계산하세요.

**요구사항:**
- 레이어 수 `num_layers = 32`, 배치 크기 `batch`, 시퀀스 길이 `seq_len = 2048`, 히든 크기 `hidden = 4096`으로 설정하세요.
- `checkpoint_every` 레이어마다 체크포인트를 저장한다고 할 때, 활성화 메모리를 계산하는 함수를 작성하세요.
- `checkpoint_every = 1, 2, 4, 8` 각 경우의 활성화 메모리를 출력하고, 체크포인팅 없는 경우(`num_layers`)와 비교하세요.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
import math

num_layers = 32
batch = 4
seq_len = 2048
hidden = 4096
bytes_per_elem = 2  # fp16

def act_memory_gb(num_layers, batch, seq_len, hidden, checkpoint_every):
    # 체크포인트 저장 레이어 수
    num_checkpoints = math.ceil(num_layers / checkpoint_every)
    # 나머지 레이어는 재계산 (메모리 불필요)
    mem = num_checkpoints * batch * seq_len * hidden * bytes_per_elem
    return mem / (1024**3)

baseline = act_memory_gb(num_layers, batch, seq_len, hidden, 1)  # 모든 레이어 저장
print(f"{'checkpoint_every':>18} {'활성화 메모리':>14} {'절감률':>10}")
print('-' * 46)
print(f"{'없음 (전체저장)':>18} {baseline:>13.2f}GB {0:>9.0f}%")
for ce in [2, 4, 8, 16]:
    mem = act_memory_gb(num_layers, batch, seq_len, hidden, ce)
    reduction = (1 - mem / baseline) * 100
    print(f"{ce:>18} {mem:>13.2f}GB {reduction:>9.1f}%")
```
</details>